# **1. Perkenalan Dataset**


Tahap pertama, Anda harus mencari dan menggunakan dataset dengan ketentuan sebagai berikut:

1. **Sumber Dataset**:  
   Dataset dapat diperoleh dari berbagai sumber, seperti public repositories (*Kaggle*, *UCI ML Repository*, *Open Data*) atau data primer yang Anda kumpulkan sendiri.


# **2. Import Library**

Pada tahap ini, Anda perlu mengimpor beberapa pustaka (library) Python yang dibutuhkan untuk analisis data dan pembangunan model machine learning atau deep learning.

In [25]:
!pip install PySastrawi
!pip install kagglehub


[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import re
import kagglehub
import pandas as pd

from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# **3. Memuat Dataset**

Pada tahap ini, Anda perlu memuat dataset ke dalam notebook. Jika dataset dalam format CSV, Anda bisa menggunakan pustaka pandas untuk membacanya. Pastikan untuk mengecek beberapa baris awal dataset untuk memahami strukturnya dan memastikan data telah dimuat dengan benar.

Jika dataset berada di Google Drive, pastikan Anda menghubungkan Google Drive ke Colab terlebih dahulu. Setelah dataset berhasil dimuat, langkah berikutnya adalah memeriksa kesesuaian data dan siap untuk dianalisis lebih lanjut.

Jika dataset berupa unstructured data, silakan sesuaikan dengan format seperti kelas Machine Learning Pengembangan atau Machine Learning Terapan

In [27]:
path = kagglehub.dataset_download("anggapurnama/twitter-dataset-ppkm", output_dir="../twitter-dataset-raw/", force_download=True)
print("Path dataset:", path)

file_path = f"{path}/INA_TweetsPPKM_Labeled_Pure.csv"
df = pd.read_csv(
    file_path,
    sep='\t',
    engine='python',
    quotechar='"',
    on_bad_lines='skip'
)

df.head()

100%|██████████| 3.58M/3.58M [01:07<00:00, 55.9kB/s]

Extracting files...
Path dataset: ../twitter-dataset-raw/


,Date,User,Tweet,sentiment
0,2022-03-31 14:32:04+00:00,pikobar_jabar,Ketahui informasi pembagian #PPKM di wilayah J...,1
1,2022-03-31 09:26:00+00:00,inewsdotid,Tempat Ibadah di Wilayah PPKM Level 1 Boleh Be...,1
2,2022-03-31 05:02:34+00:00,vdvc_talk,"Juru bicara Satgas Covid-19, Wiku Adisasmito m...",1
3,2022-03-30 14:23:10+00:00,pikobar_jabar,Ketahui informasi pembagian #PPKM di wilayah J...,1
4,2022-03-30 11:28:57+00:00,tvOneNews,Kementerian Agama menerbitkan Surat Edaran Nom...,1


# **4. Exploratory Data Analysis (EDA)**

Pada tahap ini, Anda akan melakukan **Exploratory Data Analysis (EDA)** untuk memahami karakteristik dataset.

Tujuan dari EDA adalah untuk memperoleh wawasan awal yang mendalam mengenai data dan menentukan langkah selanjutnya dalam analisis atau pemodelan.

In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23644 entries, 0 to 23643
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Date       23644 non-null  object
 1   User       23644 non-null  object
 2   Tweet      23644 non-null  object
 3   sentiment  23644 non-null  int64 
dtypes: int64(1), object(3)
memory usage: 739.0+ KB


In [29]:
df.head()

,Date,User,Tweet,sentiment
0,2022-03-31 14:32:04+00:00,pikobar_jabar,Ketahui informasi pembagian #PPKM di wilayah J...,1
1,2022-03-31 09:26:00+00:00,inewsdotid,Tempat Ibadah di Wilayah PPKM Level 1 Boleh Be...,1
2,2022-03-31 05:02:34+00:00,vdvc_talk,"Juru bicara Satgas Covid-19, Wiku Adisasmito m...",1
3,2022-03-30 14:23:10+00:00,pikobar_jabar,Ketahui informasi pembagian #PPKM di wilayah J...,1
4,2022-03-30 11:28:57+00:00,tvOneNews,Kementerian Agama menerbitkan Surat Edaran Nom...,1


In [30]:
print(df.isnull().sum())

Date         0
User         0
Tweet        0
sentiment    0
dtype: int64


In [31]:
df['sentiment'].value_counts()

sentiment
1    17706
2     3980
0     1958
Name: count, dtype: int64

In [32]:
df['length'] = df['Tweet'].apply(len)
print(df['length'].describe())

count    23644.000000
mean       175.924928
std         82.100697
min          5.000000
25%        110.000000
50%        170.000000
75%        253.000000
max        369.000000
Name: length, dtype: float64


In [33]:
print(df.duplicated().sum())
df = df.drop_duplicates()

0


# **5. Data Preprocessing**

Pada tahap ini, data preprocessing adalah langkah penting untuk memastikan kualitas data sebelum digunakan dalam model machine learning.

Jika Anda menggunakan data teks, data mentah sering kali mengandung nilai kosong, duplikasi, atau rentang nilai yang tidak konsisten, yang dapat memengaruhi kinerja model. Oleh karena itu, proses ini bertujuan untuk membersihkan dan mempersiapkan data agar analisis berjalan optimal.

Berikut adalah tahapan-tahapan yang bisa dilakukan, tetapi **tidak terbatas** pada:
1. Menghapus atau Menangani Data Kosong (Missing Values)
2. Menghapus Data Duplikat
3. Normalisasi atau Standarisasi Fitur
4. Deteksi dan Penanganan Outlier
5. Encoding Data Kategorikal
6. Binning (Pengelompokan Data)

Cukup sesuaikan dengan karakteristik data yang kamu gunakan yah. Khususnya ketika kami menggunakan data tidak terstruktur.

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)      # URL
    text = re.sub(r"@\w+", "", text)         # mention
    text = re.sub(r"#\w+", "", text)         # hashtag
    text = re.sub(r"[^a-zA-Z\s]", "", text)  # simbol
    text = re.sub(r"\s+", " ", text).strip() # spasi berlebih
    return text

df['cleaned_text'] = df['Tweet'].apply(clean_text)
df.head()

,Date,User,Tweet,sentiment,length,clean_text
0,2022-03-31 14:32:04+00:00,pikobar_jabar,Ketahui informasi pembagian #PPKM di wilayah J...,1,116,ketahui informasi pembagian di wilayah jabar b...
1,2022-03-31 09:26:00+00:00,inewsdotid,Tempat Ibadah di Wilayah PPKM Level 1 Boleh Be...,1,160,tempat ibadah di wilayah ppkm level boleh berk...
2,2022-03-31 05:02:34+00:00,vdvc_talk,"Juru bicara Satgas Covid-19, Wiku Adisasmito m...",1,302,juru bicara satgas covid wiku adisasmito menje...
3,2022-03-30 14:23:10+00:00,pikobar_jabar,Ketahui informasi pembagian #PPKM di wilayah J...,1,117,ketahui informasi pembagian di wilayah jabar b...
4,2022-03-30 11:28:57+00:00,tvOneNews,Kementerian Agama menerbitkan Surat Edaran Nom...,1,252,kementerian agama menerbitkan surat edaran nom...


In [35]:
factory = StopWordRemoverFactory()
stopwords = factory.create_stop_word_remover()

df['clean_text'] = df['clean_text'].apply(stopwords.remove)
df.head()

,Date,User,Tweet,sentiment,length,clean_text
0,2022-03-31 14:32:04+00:00,pikobar_jabar,Ketahui informasi pembagian #PPKM di wilayah J...,1,116,ketahui informasi pembagian wilayah jabar berd...
1,2022-03-31 09:26:00+00:00,inewsdotid,Tempat Ibadah di Wilayah PPKM Level 1 Boleh Be...,1,160,ibadah wilayah ppkm level berkapasitas persen ...
2,2022-03-31 05:02:34+00:00,vdvc_talk,"Juru bicara Satgas Covid-19, Wiku Adisasmito m...",1,302,juru bicara satgas covid wiku adisasmito bukbe...
3,2022-03-30 14:23:10+00:00,pikobar_jabar,Ketahui informasi pembagian #PPKM di wilayah J...,1,117,ketahui informasi pembagian wilayah jabar berd...
4,2022-03-30 11:28:57+00:00,tvOneNews,Kementerian Agama menerbitkan Surat Edaran Nom...,1,252,kementerian agama menerbitkan surat edaran nom...


In [ ]:
df = df[df['cleaned_text'].str.strip() != ""]

In [ ]:
df_final = df[['cleaned_text', 'sentiment']]

In [38]:
df_final.to_csv("../twitter-dataset-cleaned/data_clean.csv", index=False)

In [39]:
data_load = pd.read_csv("../twitter-dataset-cleaned/data_clean.csv")
data_load.head()

,clean_text,sentiment
0,ketahui informasi pembagian wilayah jabar berd...,1
1,ibadah wilayah ppkm level berkapasitas persen ...,1
2,juru bicara satgas covid wiku adisasmito bukbe...,1
3,ketahui informasi pembagian wilayah jabar berd...,1
4,kementerian agama menerbitkan surat edaran nom...,1


In [40]:
df_final.isnull().sum()

clean_text    0
sentiment     0
dtype: int64

In [41]:
df_final['sentiment'].value_counts()

sentiment
1    16152
2     3931
0     1914
Name: count, dtype: int64